---
title: Trotterized time evolution of the 2D tilted-field Ising model with LTM AI-QEM
description: Simulate Trotterized 2D tilted-field Ising dynamics with Qiskit and the LTM AI-QEM function for quantum error mitigation.
---

{/* cspell:ignore Begusic, Nayfeh, RMSE, rmse, Rosenblatt, Vovrosh, Zaletel, ecolor, edgecolor, markeredgecolor, ncols, nrows, sharex, tbegusic */}

# Trotterized time evolution of the 2D tilted-field Ising model with LTM AI-QEM

*Usage estimate: 75 seconds on a Heron r2 processor (LTM AI-QEM: 23 seconds, ZNE: 52 seconds). (NOTE: This is an estimate only. Your runtime might vary.)*


Explore the [API reference](/docs/functions/ltm-ai-qem/api-reference) for detailed function specifications.

To start using the function, follow the [LTM AI-QEM guide](/docs/functions/ltm-ai-qem).

## Background

This tutorial simulates the Trotterized time evolution of the tilted-field Ising model with the LTM AI-QEM function.

Originally conceptualized to understand ferromagnetism and quantum phase transitions in magnetic materials, the Ising model [[1]](#references) has become a cornerstone of modern many-body physics. At its fundamental level, the model describes a lattice of interacting spins, where each spin can occupy one of two states and interacts with its nearest neighbors. The real-time dynamics of two-dimensional Ising systems become increasingly difficult to simulate classically as the system size and evolution time increase, which makes them an important testbed for quantum simulation. Moreover, because the Hamiltonian is composed of local single-qubit fields and nearest-neighbor two-qubit interactions, its time evolution can be naturally represented with sequences of single- and two-qubit quantum gates. These properties make the model a useful benchmark for studying quantum dynamics on present-day hardware and for evaluating the effectiveness of quantum error mitigation methods.

This tutorial covers the following steps:

- Describe the tilted-field Ising model and its Trotterized time evolution mathematically.
- Define the backend and the qubits on which to simulate the Hamiltonian, and visualize them.
- Define the tilted-field Ising Hamiltonian with the Qiskit `SparsePauliOp` class.
- Construct the quantum circuits for the evolution of the Hamiltonian with the Qiskit `evolved_operator_ansatz` function.
- Define the observable that corresponds to the average magnetization, again with the `SparsePauliOp` class.
- Compute the evolution of the average magnetization for an increasing number of Trotter steps with the LTM AI-QEM function.
- Apply the zero-noise extrapolation (ZNE) method to the raw QPU results, to compare against the mitigation performed by the LTM AI-QEM function.
- List the values of the average magnetization computed with the sparse Pauli dynamics (SPD) algorithm, which serve as the ideal values.
- Visualize all of the results to compare the average magnetization mitigated by LTM AI-QEM with the unmitigated results, the results mitigated by ZNE, and the results from the classical SPD algorithm.

## Requirements

Install the following Python packages before running the notebook:

- Qiskit SDK v2.6.1 or later (`pip install qiskit[all]`)
- `qiskit-ibm-runtime` v0.50.0 or later (`pip install qiskit-ibm-runtime`)
- Quantum Functions Catalog v0.21.0 or later (`pip install qiskit-ibm-catalog`)
- Matplotlib v3.11.2 or later (`pip install matplotlib`)

## Setup

Authenticate using your [IBM Quantum&reg; Platform API key](https://quantum.cloud.ibm.com/), and select the LTM AI-QEM function as follows. (This snippet assumes that you already [saved your account](/docs/ibm-quantum-platform/functions-get-started#install-qiskit-functions-catalog-client) to your local environment.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# Access the LTM AI-QEM function
ltm_function = catalog.load("ltm/ai-qem")

Later in this tutorial, you use the `ltm_function` object to run the quantum circuits and get both the mitigated and unmitigated results. First, define the problem.

## The tilted-field Ising model and its time evolution

The Hamiltonian for the tilted-field Ising model is as follows:
$$
H = {g_x}\sum_{k\in{N}}{X_k} + {g_z}\sum_{k\in{N}}{Z_k} + J\sum_{\begin{matrix} {(k,l)}{\in}{E} \\ {k<l} \end{matrix}}{{Z_k}{Z_l}}
$$
where $J$ is the coupling strength, and $g_x$ and $g_z$ are the transverse and longitudinal field strengths, respectively. $X_k$ and $Z_k$ are the Pauli-X and Pauli-Z operators acting on the qubit with index $k$. $N$ is the set of indices of the qubits that evolve under the Hamiltonian. $E$ is the set of edges, each of which is a tuple of qubit indices $(k,l)$ between which two-qubit interactions are allowed. The condition $k<l$ ensures that no two-qubit interaction is repeated. The Trotterized evolution of the Hamiltonian for time $t$ with $r$ Trotter steps, each of time $\Delta{t}$ (that is, $t=r\Delta{t}$), is as follows:

$$
\begin{align}
e^{iHt}
&{\quad}={\quad}
e^{{i}\bigg({g_x}\sum_{k\in{N}}{X_k} + {g_z}\sum_{k\in{N}}{Z_k} + J\sum_{\begin{matrix} {(k,l)}{\in}{E} \\ {k<l} \end{matrix}}{{Z_k}{Z_l}}\bigg){t}}
\\\\ &{\quad}\approx{\quad}
\Bigg[
\Bigg(e^{{i}\big({g_x}\sum_{k\in{N}}{X_k}\big){\Delta{t}}}\Bigg)
\Bigg(e^{{i}\big({g_z}\sum_{k\in{N}}{Z_k}\big){\Delta{t}}}\Bigg)
\Bigg(e^{{i}\bigg(J\sum_{\begin{matrix} {(k,l)}{\in}{E} \\ {k<l} \end{matrix}}{{Z_k}{Z_l}}\bigg){\Delta{t}}}\Bigg)
\Bigg]^r
\\\\ &{\quad}={\quad}
\Bigg[
\bigg(\prod_{k\in{N}}{e^{{i}g_x{X_k}{\Delta{t}}}}\bigg)
\bigg(\prod_{k\in{N}}{e^{{i}g_z{Z_k}{\Delta{t}}}}\bigg)
\bigg(\prod_{\begin{matrix} {(k,l)}{\in}{E} \\ {k<l} \end{matrix}}{e^{{i}J{Z_k}{Z_l}{\Delta{t}}}}\bigg)
\Bigg]^r
\end{align}
$$

You return to this evolution operator later, when you construct the quantum circuit that represents the evolution. Next, specify the qubits on which to perform the evolution.

## Step 1: Specify and visualize the backend and the problem

The LTM AI-QEM function currently supports Heron processors, so this tutorial simulates the Ising model dynamics on a Heron backend. A graph represents the backend: its nodes represent the qubits, and its edges represent the pairs of qubits between which two-qubit gates can be applied. Extract the number of qubits and the set of all edges, which you need to construct the quantum circuit.

In [7]:
backend = catalog.backend("ibm_fez")

backend_num_nodes = backend.num_qubits
backend_nodes = list(range(backend_num_nodes))
# Get the list of edges in the graph (allowed two-qubit interactions)
backend_edges = list(backend.coupling_map.get_edges())


print("Backend specification:")
print(f"\tName: {backend.name}")
print(f"\tNumber of qubits: {backend_num_nodes}")
print(f"\tQubit indices: {backend_nodes}")
print(f"\tTwo-qubit interactions: {backend_edges}")

Backend specification:
	Name: ibm_fez
	Number of qubits: 156
	Qubit indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]
	Two-qubit interactions: [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (3, 16), (4, 3), (4, 5), (5, 4), (5, 6), (6, 5), (6, 7), (7, 6), (7, 8), (7, 17), (8, 7), (8, 9), (9, 8), (9, 10), (10, 9), (10, 11), (11, 10), (11, 12), (11, 18), (12, 1

Now, set the qubits on which to run the simulation. Define a list, `problem_nodes`, that contains the indices of the qubits on which to perform the evolution, and a list, `problem_edges`, that contains the two-qubit interactions between those qubits.

In [8]:
# Define the problem_nodes and the problem_edges
problem_nodes = list(range(56))
problem_edges = [
    (i, j)
    for i, j in backend_edges
    if i in problem_nodes and j in problem_nodes and i < j
]


print("Problem Specification:")
print(f"\tNumber of qubits: {len(problem_nodes)}")
print(f"\tQubit indices: {problem_nodes}")
print(f"\tTwo-qubit interaction pairs: {problem_edges}")

Problem Specification:
	Number of qubits: 56
	Qubit indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]
	Two-qubit interaction pairs: [(0, 1), (1, 2), (2, 3), (3, 4), (3, 16), (4, 5), (5, 6), (6, 7), (7, 8), (7, 17), (8, 9), (9, 10), (10, 11), (11, 12), (11, 18), (12, 13), (13, 14), (14, 15), (15, 19), (16, 23), (17, 27), (18, 31), (19, 35), (20, 21), (21, 22), (21, 36), (22, 23), (23, 24), (24, 25), (25, 26), (25, 37), (26, 27), (27, 28), (28, 29), (29, 30), (29, 38), (30, 31), (31, 32), (32, 33), (33, 34), (33, 39), (34, 35), (36, 41), (37, 45), (38, 49), (39, 53), (40, 41), (41, 42), (42, 43), (43, 44), (44, 45), (45, 46), (46, 47), (47, 48), (48, 49), (49, 50), (50, 51), (51, 52), (52, 53), (53, 54), (54, 55)]


### Visualize the problem on the backend

Visualize the problem nodes as a subgraph of the backend nodes to confirm that the selected subgraph is 2D.

In [9]:
from qiskit.visualization import plot_coupling_map


plot_coupling_map(
    num_qubits=backend.num_qubits,
    qubit_coordinates=getattr(
        backend.configuration(), "qubit_coordinates", None
    ),
    coupling_map=backend.configuration().coupling_map,
    figsize=(12, 12),
    qubit_color=[
        "#4CAF50" if i in set(problem_nodes) else "#DDDDDD"
        for i in backend.coupling_map.graph.node_indices()
    ],
    qubit_size=90,
    line_width=2,
    font_size=40,
)

<Image src="/docs/images/tutorials/trotterized-time-evolution-ltm/extracted-outputs/f1251e76-0.avif" alt="Output of the previous code cell" />

## Step 2: Define the Hamiltonian

Define the Ising Hamiltonian with the Qiskit `SparsePauliOp` class. To do this, specify the angles `theta_x` $(\theta_x)$, `theta_z` $(\theta_z)$, and `theta_zz` $(\theta_{zz})$, which are related to the transverse field strength (`g_x`), the longitudinal field strength (`g_z`), and the Ising coupling strength (`J`), respectively. Specifically, $\theta_x=2g_x\Delta{t}$, $\theta_z=2g_z\Delta{t}$, and $\theta_{zz}=2J\Delta{t}$.

Use the `SparsePauliOp.from_sparse_list` method to instantiate the transverse field Hamiltonian (`Hx`), the longitudinal field Hamiltonian (`Hz`), and the coupling Hamiltonian (`Hzz`). The `SparsePauliOp.from_sparse_list` method takes as input a list of tuples. Each tuple contains a Pauli string, the indices of the qubits that the Pauli operators in the string act on, and the coefficient of the Pauli string. For more details, see the [`SparsePauliOp`](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp) API documentation. The qubit indices come from the `problem_nodes` and `problem_edges` objects defined earlier. The full tilted-field Ising Hamiltonian is the sum of these three Hamiltonians.

In [10]:
from qiskit.quantum_info import SparsePauliOp

theta_x = 0.53
theta_z = 0.1
theta_zz = 1.0

g_x = theta_x / 2
g_z = theta_z / 2
J = theta_zz / 2

Hx = SparsePauliOp.from_sparse_list(
    [("X", [node], theta_x / 2) for node in problem_nodes],
    num_qubits=backend_num_nodes,
)
Hz = SparsePauliOp.from_sparse_list(
    [("Z", [node], theta_z / 2) for node in problem_nodes],
    num_qubits=backend_num_nodes,
)
Hzz = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], theta_zz / 2) for i, j in problem_edges],
    num_qubits=backend_num_nodes,
)
H = Hx + Hz + Hzz

print(f"Indices of qubits to be evolved(problem_nodes):\n{problem_nodes}\n")
print(f"Transverse Field Hamiltonian(Hx):\n{Hx}\n")
print(f"Longitudinal Field Hamiltonian(Hz):\n{Hz}\n")
print(f"Coupling Hamiltonian(Hzz):\n{Hzz}\n")
print(f"Tilted Field Ising Hamiltonian(H = Hx + Hz + Hzz):\n{H}\n")

Indices of qubits to be evolved(problem_nodes):
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]

Transverse Field Hamiltonian(Hx):
SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIX', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIXIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

## Step 3: Construct the Trotterized quantum circuit

Now, construct the quantum circuit that corresponds to the Trotterized evolution of the qubits in `problem_nodes` under the tilted-field Ising model. To simulate the time evolution, construct a sequence of quantum circuits with an increasing number of Trotter steps (for example, 1, 2, and 3 steps). Set the number of Trotter steps, `num_trotter_steps`, to 10, and create a range, `trotter_steps`, that contains the numbers from 1 to `num_trotter_steps`.

To construct the quantum circuit for a specific number of Trotter steps, use the Qiskit [`evolved_operator_ansatz`](/docs/api/qiskit/qiskit.circuit.library.evolved_operator_ansatz) function. This function takes as input the tilted-field Ising Hamiltonian, defined earlier as a `SparsePauliOp` object, and outputs the quantum circuit that corresponds to the evolution. Store the circuits in a dictionary, `trotter_circ_dict`, whose keys are the numbers of Trotter steps and whose values are the corresponding quantum circuits. The code inserts barriers between Trotter steps only to make the circuit diagram easier to read. You can remove the code that inserts the barriers.

Note that `evolved_operator_ansatz` outputs a parameterized quantum circuit with a parameter `t` that takes the value of $\Delta{t}$ in the evolution operator. Because the definitions of the `theta` angles already include $\Delta{t}$, set all parameters to `t=1`.

In [11]:
from qiskit.circuit.library import evolved_operator_ansatz
from qiskit.circuit import Barrier, CircuitInstruction


def generate_trotter_step_circuit(ham, step):
    circuit = evolved_operator_ansatz(ham, reps=step, insert_barriers=False)  # type: ignore
    circuit.assign_parameters([1] * circuit.num_parameters, inplace=True)

    barrier_positions = [n * len(ham) for n in range(1, step + 1)]
    barrier_op = Barrier(circuit.num_qubits)
    barrier_instruction = CircuitInstruction(
        operation=barrier_op, qubits=circuit.qubits
    )
    for barrier_pos in sorted(barrier_positions, reverse=True):
        circuit.data.insert(barrier_pos, barrier_instruction)

    return circuit


num_trotter_steps = 10
trotter_steps = range(1, num_trotter_steps + 1)

trotter_circ_dict = {}
for step in trotter_steps:
    circuit = generate_trotter_step_circuit(H, step)  # type: ignore
    trotter_circ_dict[step] = circuit

# Visualize one of the circuit
trotter_circ_dict[3].draw("mpl", idle_wires=False, fold=-1)

<Image src="/docs/images/tutorials/trotterized-time-evolution-ltm/extracted-outputs/8c0f5da2-0.avif" alt="Output of the previous code cell" />

The exponentials of the Pauli operators $X$, $Z$, and ${Z}{Z}$ are the following rotation gates:
$$
R_x(\theta_x)=e^{-i{\frac{\theta_x}{2}}X}
\quad;\quad
R_z(\theta_z)=e^{-i{\frac{\theta_z}{2}}Z}
\quad;\quad
R_{zz}(\theta_{zz})=e^{-i{\frac{\theta_{zz}}{2}}ZZ}
$$
So the Trotterized evolution becomes:
$$
e^{iHt} \approx \Bigg[
\bigg(\prod_{k\in{N}}{R_x(\theta_x)}\bigg)
\bigg(\prod_{k\in{N}}{R_z(\theta_z)}\bigg)
\bigg(\prod_{\begin{matrix} {(k,l)}{\in}{E} \\ {k<l} \end{matrix}}{R_{zz}(\theta_{zz})}\bigg)
\Bigg]^r
$$
The previous circuit diagram shows the repeated pattern of the circuit. Each step contains $R_x(\theta_x=0.53)$ and $R_z(\theta_z=0.1)$ gates on each qubit, and an $R_{zz}(\theta_{zz}=1)$ gate between each pair of qubits where two-qubit interactions are allowed.

## Step 4: Construct the observable for average magnetization

The observable for the average magnetization is as follows:
$$
H_M = \frac{1}{|N|}\Bigg(\sum_{k\in{N}}{Z_k}\Bigg)
$$
where $|N|$ is the size of the set $N$. Use the `SparsePauliOp` class again to define the observable `Hm`, which corresponds to the average magnetization.

In [ ]:
Hm = SparsePauliOp.from_sparse_list(
    [("Z", [node], 1 / len(problem_nodes)) for node in problem_nodes],
    num_qubits=backend_num_nodes,
)

print(f"The observable corresponding to average magnetization(Hm):\n{Hm}")

The observable corresponding to average magnetization(Hm):
SparsePauliOp(['IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZI', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIZIIII', 'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

## Step 5: Compute the expectation value with the LTM AI-QEM function

Compute the expectation value of the average magnetization observable, `Hm`, for each circuit in `trotter_circ_dict`. To do this, transpile each circuit, apply the layout of the transpiled circuit to `Hm`, and create a list of primitive unified blocs (PUBs). Each PUB contains the transpiled circuit for a specific number of Trotter steps and the corresponding observable.

In [13]:
from qiskit.transpiler import generate_preset_pass_manager

# Generate pass manager to transpile
pm = generate_preset_pass_manager(
    backend=backend,
    initial_layout=backend_nodes,
    optimization_level=3,
    seed_transpiler=42,
)

# Construct ISA PUBs
isa_pubs = []
for circuit in trotter_circ_dict.values():
    isa_circuit = pm.run(circuit)
    isa_Hm = Hm.apply_layout(isa_circuit.layout)
    isa_pubs.append((isa_circuit, isa_Hm))

Then pass the list of PUBs to `ltm_function`, which you defined at the beginning of the tutorial. You can either pass transpiled circuits with the layout applied to the corresponding observables, as shown in the following code, or pass the transpilation options through `options`. If you pass the transpilation options, the function transpiles the circuits and applies the layout to the observables.

In [ ]:
# Submit the job for execution to ltm_function
job = ltm_function.run(  # type: ignore
    primitive="estimator",
    backend_name=backend.name,
    pubs=isa_pubs,
    transpiled=True,
    options={},
)
print("Job ID: ", job.job_id)

Job ID:  cf493f78-2ee0-4129-8ac1-ba531ce67e77


In [23]:
print("Current Status: ", job.status())

Current Status:  DONE


In [24]:
# Wait for the job to complete (blocking code)
results = job.result()

# Extracting the expectation values
noisy_evs, noisy_stds = [], []
mitigated_evs, mitigated_stds = [], []

for pub_index in range(len(results)):
    mitigated_evs.append(results[pub_index].data.evs.item())
    mitigated_stds.append(results[pub_index].data.stds.item())
    noisy_evs.append(results[pub_index].data.noisy_evs.item())
    noisy_stds.append(results[pub_index].data.noisy_stds.item())


print(f"Noisy Expectation Value:\n\t{noisy_evs}")
print(f"Noisy Standard Deviations:\n\t{noisy_stds}")
print(f"Mitigated Expectation Value:\n\t{mitigated_evs}")
print(f"Mitigated Standard Deviations:\n\t{mitigated_stds}")

Noisy Expectation Value:
	[0.8092386738777213, 0.7118844012886482, 0.6423511643594512, 0.5885756231821132, 0.5375291128012729, 0.48870442988150503, 0.4538531564760261, 0.4207090656884491, 0.41135317419835765, 0.3896190603795144]
Noisy Standard Deviations:
	[0.011531391052123982, 0.01312993527331503, 0.01416826960273852, 0.014593899733964518, 0.015095683009088272, 0.016037710150720686, 0.01527758519049225, 0.01601427030364386, 0.015951107170769738, 0.01573650957643109]
Mitigated Expectation Value:
	[0.975075399350835, 0.870229902306979, 0.8293153706649651, 0.8074930473397531, 0.7792975357034224, 0.7509270294820537, 0.7318380664289472, 0.7082185676338562, 0.7041947038040618, 0.6957669116779336]
Mitigated Standard Deviations:
	[0.013843007049931001, 0.016056708607027022, 0.01846749673601081, 0.02051612386065825, 0.022309193683744757, 0.025284343136726888, 0.025571029192604372, 0.0277725053279589, 0.028263287822675925, 0.029997863650503213]


## Compare against ZNE

Compute the average magnetization with the zero-noise extrapolation (ZNE) method available in `qiskit-ibm-runtime`. Define the `EstimatorV2` primitive to compute the expectation values, and enable ZNE error mitigation with `estimator.options`, as shown in the following code.

In [25]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = ("exponential", "linear")

estimator_job = estimator.run(isa_pubs)

  estimator = Estimator(mode=backend)


In [26]:
print("IQP Job ID: ", estimator_job.job_id())

IQP Job ID:  dbevnj7sstms73fcsb30


In [36]:
print("Current Status: ", estimator_job.status())

Current Status:  DONE


In [37]:
# Extracting the result
estimator_job_result = estimator_job.result()

zne_evs, zne_stds = [], []
for pub_index in range(len(estimator_job_result)):
    zne_evs.append(estimator_job_result[pub_index].data.evs.item())
    zne_stds.append(estimator_job_result[pub_index].data.stds.item())

print(f"ZNE Expectation Values:\n\t{zne_evs}")
print(f"ZNE Standard Deviations:\n\t{zne_stds}")

ZNE Expectation Values:
	[0.8214756326353143, 0.7224173809715188, 0.6659171810960559, 0.6115965411647895, 0.5796109662658784, 0.5170033717427326, 0.46295669649028703, 0.4471263382869598, 0.4103592730051738, 0.3844589789914816]
ZNE Standard Deviations:
	[0.03747105567623526, 0.03001103933917704, 0.03546971884719226, 0.038776684873607525, 0.044887225137344555, 0.04191094125537236, 0.03786578543486445, 0.040189446627697405, 0.03660628916887856, 0.026134217954006264]


## Sparse Pauli dynamics results

This tutorial uses the classical sparse Pauli dynamics (SPD) algorithm [[2, 3]](#references) to evaluate the average magnetization. SPD was previously applied to simulate the kicked transverse-field Ising model experiment in [[4]](#references). This tutorial treats the SPD results as ideal, and compares the results from the LTM AI-QEM function against them. To keep this tutorial simple, it doesn't include the code that generates the SPD results. Instead, the results are provided in the `spd_results` list. The implementation of the SPD algorithm by its authors is available in [[5]](#references).

In [38]:
# Expectation values computed by the Sparse Pauli Dynamics Algorithm
spd_results = [
    0.8628070705147611,
    0.8360142130230876,
    0.8608833252602561,
    0.8249005552138493,
    0.8052255340483068,
    0.8258743166180563,
    0.794030079205438,
    0.7740663098219183,
    0.7753111866660567,
    0.7545411193863925,
]

## Compare the results

To compare the mitigation methods, plot the results from LTM AI-QEM, ZNE, and the unmitigated hardware execution against the SPD reference for each number of Trotter steps. The code also prints a table of key performance metrics for a quantitative comparison.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def compute_improvement_metrics(
    spd_results, mitigated_evs, zne_evs, noisy_evs
):
    """Quantify how much each mitigation method improves on the unmitigated result.

    Error for each method is |SPD - method| at every Trotter step (SPD is ideal).

    Returns a dict keyed by method name with:
      - mae:               mean absolute error vs SPD
      - rmse:              root mean squared error vs SPD
      - mean_improvement:  mean(err_unmitigated - err_method)   (absolute units, >0 is better)
      - pct_improvement:   100 * (MAE_unmitigated - MAE_method) / MAE_unmitigated
      - error_reduction_factor: MAE_unmitigated / MAE_method    (>1 is better)
      - frac_steps_improved:   fraction of steps where the method beats unmitigated
    """
    spd = np.asarray(spd_results, dtype=float)
    errs = {
        "Unmitigated": np.abs(spd - np.asarray(noisy_evs, dtype=float)),
        "ZNE": np.abs(spd - np.asarray(zne_evs, dtype=float)),
        "LTM AI-QEM": np.abs(spd - np.asarray(mitigated_evs, dtype=float)),
    }
    base = errs["Unmitigated"]
    base_mae = base.mean()
    eps = 1e-12

    metrics = {}
    for name, e in errs.items():
        mae = e.mean()
        metrics[name] = {
            "mae": mae,
            "rmse": float(np.sqrt(np.mean(e**2))),
            "mean_improvement": float((base - e).mean()),
            "pct_improvement": float(
                100 * (base_mae - mae) / max(base_mae, eps)
            ),
            "error_reduction_factor": float(base_mae / max(mae, eps)),
        }
    return metrics


def plot_trotter_results(
    trotter_steps,
    spd_results,
    mitigated_evs,
    mitigated_errs,
    zne_evs,
    zne_errs,
    noisy_evs,
    noisy_errs,
    title="",
    print_summary=True,
):
    """Plot magnetization versus Trotter steps, and the absolute difference from SPD for each method.

    Top: average magnetization for each method.
    Bottom: grouped bars of |SPD - method|, with mean improvement annotated.

    Returns a dict of improvement metrics (see compute_improvement_metrics).
    """
    spd = np.asarray(spd_results)
    steps = np.asarray(trotter_steps)

    metrics = compute_improvement_metrics(
        spd_results, mitigated_evs, zne_evs, noisy_evs
    )

    series = [
        (spd_results, None, "Sparse Pauli Dynamics", "*", "green"),
        (
            mitigated_evs,
            mitigated_errs,
            "Mitigation by LTM AI-QEM",
            "P",
            "royalblue",
        ),
        (zne_evs, zne_errs, "Mitigation by ZNE", "^", "darkorange"),
        (noisy_evs, noisy_errs, "Unmitigated", "X", "red"),
    ]

    fig, (ax1, ax2) = plt.subplots(
        nrows=2, ncols=1, figsize=(12, 10), sharex=True, height_ratios=[2, 1]
    )

    for values, errs, label, marker, color in series:
        ax1.errorbar(
            steps,
            values,
            yerr=errs,
            label=label,
            marker=marker,
            color=color,
            markeredgecolor="black",
            markeredgewidth=0.5,
            ecolor="black",
            capsize=5,
        )

    ax1.set_ylabel("Average Magnetization")
    ax1.set_title(title)
    ax1.grid(alpha=0.3)
    ax1.legend()

    # Bottom: grouped bars with MAE and % improvement in the legend labels
    bars = [
        (mitigated_evs, "LTM AI-QEM", "royalblue"),
        (zne_evs, "ZNE", "darkorange"),
        (noisy_evs, "Unmitigated", "red"),
    ]
    n = len(bars)
    width = 0.8 / n
    for i, (values, key, color) in enumerate(bars):
        offset = (i - (n - 1) / 2) * width
        m = metrics[key]
        if key == "Unmitigated":
            label = f"|SPD - {key}| (MAE={m['mae']:.3f})"
        else:
            label = (
                f"|SPD - {key}| (MAE={m['mae']:.3f}, "
                f"{m['pct_improvement']:+.1f}% vs unmitigated)"
            )
        ax2.bar(
            steps + offset,
            np.abs(spd - np.asarray(values)),
            width=width,
            label=label,
            color=color,
            edgecolor="black",
            linewidth=0.5,
        )
        if key != "Unmitigated":
            pass
        ax2.axhline(
            m["mae"], color=color, linestyle="--", linewidth=1, alpha=0.8
        )

    ax2.set_xticks(steps)
    ax2.set_xlabel("Trotter Steps")
    ax2.set_ylabel("|SPD - method|")
    ax2.set_title(
        "Difference from SPD (closer to 0 is better; dashed = mean error)"
    )
    ax2.grid(alpha=0.3, axis="y")
    ax2.legend()

    fig.tight_layout()
    plt.show()

    if print_summary:
        print(
            f"{'Method':<12}{'MAE':>10}{'RMSE':>10}{'Mean impr.':>12}"
            f"{'% impr.':>10}{'Factor':>9}"
        )
        for name, m in metrics.items():
            print(
                f"{name:<12}{m['mae']:>10.4f}{m['rmse']:>10.4f}"
                f"{m['mean_improvement']:>12.4f}{m['pct_improvement']:>9.1f}%"
                f"{m['error_reduction_factor']:>9.2f}"
            )

    return None

In [16]:
plot_trotter_results(
    trotter_steps,
    spd_results,
    mitigated_evs,
    mitigated_stds,
    zne_evs,
    zne_stds,
    noisy_evs,
    noisy_stds,
)

<Image src="/docs/images/tutorials/trotterized-time-evolution-ltm/extracted-outputs/84a5d8f2-0.avif" alt="Output of the previous code cell" />

Method             MAE      RMSE  Mean impr.   % impr.   Factor
Unmitigated     0.2660    0.2852      0.0000      0.0%     1.00
ZNE             0.2491    0.2704      0.0169      6.4%     1.07
LTM AI-QEM      0.0554    0.0617      0.2106     79.2%     4.80


The LTM AI-QEM results closely follow the SPD results for all Trotter steps. LTM AI-QEM reduces the mean absolute error (MAE) to 0.055, compared with 0.249 for ZNE and 0.266 for the unmitigated results. This is a 79.2% improvement over the unmitigated results, whereas ZNE provides a 6.4% improvement. The corresponding root mean squared error (RMSE) values are 0.062 for LTM AI-QEM, 0.270 for ZNE, and 0.285 for the unmitigated results. LTM AI-QEM achieves this improvement without additional QPU time, because the correction is applied during post-processing.

## References

[1] [J. Vovrosh et al., “Simulating dynamics of the two-dimensional transverse-field Ising model: A comparative study of large-scale classical numerics,” Physical Review Research 8, 023311 (2026).](https://journals.aps.org/prresearch/abstract/10.1103/x4gs-g63p)

[2] [T. Begusic, J. Gray, G. K. Chan, Sci. Adv. 10, eadk4321 (2024). arXiv:2308.05077 (2023).](https://arxiv.org/abs/2308.05077)

[3] [T. Begusic, G. K. Chan, PRX Quantum 6, 020302 (2025).](https://link.aps.org/doi/10.1103/PRXQuantum.6.020302)

[4] [Y. Kim, A. Eddins, S. Anand, K. X. Wei, E. Van Den Berg, S. Rosenblatt, H. Nayfeh, Y. Wu, M. Zaletel, K. Temme, and A. Kandala, “Evidence for the utility of quantum computing before fault tolerance,” Nature 618, 500 (2023).](https://doi.org/10.1038/s41586-023-06096-3)

[5] [GitHub tbegusic/spd](https://github.com/tbegusic/spd)